# Randevuya Gelmeme (No-Show) Tahmini — düzenlenmiş sürüm
Hedef: `label_noshow`, metrik: PR-AUC (`average_precision_score`).

**Bu sürümde yapılan düzeltmeler**
1. Veri klasörü otomatik bulunuyor (Kaggle `/kaggle/input/...` veya not defterinin klasörü) → `FileNotFoundError` gitti.
2. Target encoding artık modelle **aynı fold'ları** (StratifiedGroupKFold, grup = `patient_id`) kullanıyor → doğrulama skorundaki sızıntı (leakage) giderildi.
3. Ham `patient_id` modelden çıkarıldı; grup CV ile hep yeni hastalar üzerinde doğrulandığı için yalnızca ezberlemeye yarıyordu (frekansı ve diğer hasta özellikleri duruyor).
4. CatBoost isteğe bağlı: kuruluysa eğitiliyor, değilse atlanıyor. Ensemble ağırlıkları **eğitilen modellere göre normalize ediliyor**; eskiden sıfırlardan oluşan CatBoost tahmini ensemble'a %30 ağırlıkla giriyor ve boş bir CatBoost submission dosyası yazılıyordu.
5. XGBoost'a early stopping eklendi.
6. Submission, sıraya güvenmek yerine `appointment_id` üzerinden eşleştiriliyor ve `sample_submission` sütun adları korunuyor.
7. Çıplak `except:` → `except ImportError:`; `!pip install` koşullu hale geldi.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import glob
import os

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold

# StratifiedGroupKFold (sklearn >= 1.0) varsa onu kullanalım, yoksa fallback yapacağız
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except ImportError:
    HAS_SGKF = False

# CatBoost isteğe bağlı: kurulu değilse atlanır, kod yine çalışır
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost bulunamadı; yalnızca LightGBM + XGBoost kullanılacak. "
          "Kurmak için: !pip install catboost  (sonra çekirdeği yeniden başlatın)")

RANDOM_STATE = 42
TARGET = "label_noshow"
N_SPLITS = 5

In [ ]:
# Veri klasörünü otomatik bul: önce Kaggle input, sonra not defterinin klasörü
def find_data_dir(marker="appointments_train.csv"):
    candidates = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    candidates += glob.glob(f"./**/{marker}", recursive=True)
    if not candidates:
        raise FileNotFoundError(
            f"{marker} bulunamadı. Kaggle'da veri setini 'Add Input' ile ekleyin "
            "ya da CSV dosyalarını not defteriyle aynı klasöre koyun."
        )
    return os.path.dirname(candidates[0])

DATA_DIR = find_data_dir()
print("Veri klasörü:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, "appointments_train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "appointments_test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

patients = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"))
clinics = pd.read_csv(os.path.join(DATA_DIR, "clinics.csv"))

print("appointments_train:", train.shape)
print("appointments_test :", test.shape)
print("patients          :", patients.shape)
print("clinics           :", clinics.shape)
print("sample_submission :", sample_sub.shape)

In [ ]:
if "specialty" in clinics.columns and "clinic_specialty" not in clinics.columns:
    clinics = clinics.rename(columns={"specialty": "clinic_specialty"})

# Join anahtarlarının tekil olduğunu kontrol et (aksi halde merge satır çoğaltır)
assert patients["patient_id"].is_unique, "patients.csv içinde tekrar eden patient_id var"
assert clinics["clinic_id"].is_unique, "clinics.csv içinde tekrar eden clinic_id var"

n_train, n_test = len(train), len(test)

train = train.merge(patients, on="patient_id", how="left")
test = test.merge(patients, on="patient_id", how="left")

train = train.merge(clinics, on="clinic_id", how="left")
test = test.merge(clinics, on="clinic_id", how="left")

assert len(train) == n_train and len(test) == n_test, "Merge sonrası satır sayısı değişti"

print("\nAfter merge:")
print("train:", train.shape)
print("test :", test.shape)

In [ ]:
def add_time_features(df):
    df = df.copy()

     # datetime dönüşümünü garanti et
    if "appointment_datetime" in df.columns:
        df["appointment_datetime"] = pd.to_datetime(
            df["appointment_datetime"], errors="coerce"
        )

    if "booking_datetime" in df.columns:
        df["booking_datetime"] = pd.to_datetime(
            df["booking_datetime"], errors="coerce"
        )
    # -------------------------
    # appointment datetime
    # -------------------------
    if "appointment_datetime" in df.columns:
        df["appointment_year"] = df["appointment_datetime"].dt.year
        df["appointment_month"] = df["appointment_datetime"].dt.month
        df["appointment_day"] = df["appointment_datetime"].dt.day
        df["appointment_hour"] = df["appointment_datetime"].dt.hour
        df["appointment_minute"] = df["appointment_datetime"].dt.minute
        df["appointment_dow"] = df["appointment_datetime"].dt.weekday
        df["appointment_weekofyear"] = (
            df["appointment_datetime"].dt.isocalendar().week.astype("float")
        )
        df["is_weekend"] = df["appointment_dow"].isin([5, 6]).astype(int)
        df["is_monday"] = (df["appointment_dow"] == 0).astype(int)
        df["is_friday"] = (df["appointment_dow"] == 4).astype(int)
        df["is_morning"] = df["appointment_hour"].between(6, 11).astype(int)
        df["is_lunch_hour"] = df["appointment_hour"].isin([12, 13]).astype(int)
        df["is_evening"] = (df["appointment_hour"] >= 17).astype(int)

        df["hour_group"] = pd.cut(
            df["appointment_hour"],
            bins=[-1, 8, 12, 16, 24],
            labels=["early_morning", "morning", "afternoon", "evening"]
        ).astype("object")

    # -------------------------
    # booking datetime
    # -------------------------
    if "booking_datetime" in df.columns:
        df["booking_year"] = df["booking_datetime"].dt.year
        df["booking_month"] = df["booking_datetime"].dt.month
        df["booking_day"] = df["booking_datetime"].dt.day
        df["booking_hour"] = df["booking_datetime"].dt.hour
        df["booking_dow"] = df["booking_datetime"].dt.weekday
        df["booking_is_weekend"] = df["booking_dow"].isin([5, 6]).astype(int)

    # -------------------------
    # datetime differences
    # -------------------------
    if "appointment_datetime" in df.columns and "booking_datetime" in df.columns:
        diff = df["appointment_datetime"] - df["booking_datetime"]
        df["days_diff"] = diff.dt.days
        df["hours_diff_calc"] = diff.dt.total_seconds() / 3600.0
        df["minutes_diff_calc"] = diff.dt.total_seconds() / 60.0

    # lead_time_hours varsa türevleri
    if "lead_time_hours" in df.columns:
        df["lead_time_days"] = df["lead_time_hours"] / 24.0
        df["lead_time_weeks"] = df["lead_time_hours"] / (24.0 * 7.0)
        df["short_lead"] = (df["lead_time_hours"] <= 24).astype(int)
        df["very_short_lead"] = (df["lead_time_hours"] <= 6).astype(int)
        df["long_lead"] = (df["lead_time_hours"] >= 72).astype(int)

    return df


def add_sms_features(df):
    df = df.copy()

    if "sms_lead_hours" in df.columns:
        df["sms_missing"] = df["sms_lead_hours"].isna().astype(int)
        df["has_sms"] = df["sms_lead_hours"].notna().astype(int)
        df["sms_lead_days"] = df["sms_lead_hours"] / 24.0
        df["late_sms"] = (df["sms_lead_hours"] <= 6).fillna(0).astype(int)
        df["early_sms"] = (df["sms_lead_hours"] >= 24).fillna(0).astype(int)

    return df


def add_ratio_and_interaction_features(df):
    df = df.copy()

    # Bazı alanlar yoksa hata vermesin diye kontrollü gidiyoruz
    if {"lead_time_hours", "wait_mins_est"}.issubset(df.columns):
        df["time_pressure"] = df["lead_time_hours"] / (df["wait_mins_est"] + 1)
        df["wait_x_lead"] = df["wait_mins_est"] * df["lead_time_hours"]

    if {"prior_noshow_rate", "ses_score", "distance_km"}.issubset(df.columns):
        df["behavior_score"] = (
            df["prior_noshow_rate"] * 3
            + df["ses_score"]
            + (df["distance_km"] / 20.0)
        )

    if {"distance_km", "lead_time_hours"}.issubset(df.columns):
        df["distance_x_lead"] = df["distance_km"] * df["lead_time_hours"]
        df["distance_per_lead"] = df["distance_km"] / (df["lead_time_hours"] + 1)

    if {"ses_score", "prior_noshow_rate"}.issubset(df.columns):
        df["ses_x_noshowhist"] = df["ses_score"] * df["prior_noshow_rate"]

    if {"wait_mins_est", "appointment_hour"}.issubset(df.columns):
        df["wait_x_hour"] = df["wait_mins_est"] * df["appointment_hour"]

    if {"lead_time_hours", "appointment_hour"}.issubset(df.columns):
        df["lead_x_hour"] = df["lead_time_hours"] * df["appointment_hour"]

    if {"lead_time_days", "appointment_dow"}.issubset(df.columns):
        df["lead_x_dow"] = df["lead_time_days"] * df["appointment_dow"]

    if {"distance_km", "wait_mins_est"}.issubset(df.columns):
        df["distance_x_wait"] = df["distance_km"] * df["wait_mins_est"]

    return df


def add_geographical_features(df):
    df = df.copy()

    if {"residence_lat", "clinic_lat"}.issubset(df.columns):
        df["lat_diff"] = (df["residence_lat"] - df["clinic_lat"]).abs()

    if {"residence_lon", "clinic_lon"}.issubset(df.columns):
        df["lon_diff"] = (df["residence_lon"] - df["clinic_lon"]).abs()

    if {"lat_diff", "lon_diff"}.issubset(df.columns):
        df["geo_manhattan"] = df["lat_diff"] + df["lon_diff"]

    return df


def add_combination_categoricals(df):
    df = df.copy()

    if {"clinic_id", "appointment_dow"}.issubset(df.columns):
        df["clinic_dow_combo"] = (
            df["clinic_id"].astype(str) + "_" + df["appointment_dow"].astype(str)
        )

    if {"booking_channel", "appointment_hour"}.issubset(df.columns):
        df["channel_hour_combo"] = (
            df["booking_channel"].astype(str) + "_" + df["appointment_hour"].astype(str)
        )

    if {"clinic_specialty", "appointment_dow"}.issubset(df.columns):
        df["specialty_dow_combo"] = (
            df["clinic_specialty"].astype(str) + "_" + df["appointment_dow"].astype(str)
        )

    if {"clinic_specialty", "hour_group"}.issubset(df.columns):
        df["specialty_hour_combo"] = (
            df["clinic_specialty"].astype(str) + "_" + df["hour_group"].astype(str)
        )

    return df


def add_all_features(df):
    df = df.copy()
    df = add_time_features(df)
    df = add_sms_features(df)
    df = add_geographical_features(df)
    df = add_ratio_and_interaction_features(df)
    df = add_combination_categoricals(df)
    return df


def add_frequency_encoding(train_df, test_df, cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cols:
        if col in train_df.columns and col in test_df.columns:
            freq = train_df[col].astype(str).value_counts(dropna=False)
            train_df[f"{col}_freq"] = train_df[col].astype(str).map(freq).fillna(0)
            test_df[f"{col}_freq"] = test_df[col].astype(str).map(freq).fillna(0)

    return train_df, test_df


def kfold_target_encode(train_df, test_df, col, target, splits, alpha=20):
    """
    OOF target encoding (smoothing'li).
    - train tarafı: her satır, kendi fold'u DIŞINDAKİ verilerle kodlanır.
    - test tarafı : tüm train ile kodlanır.
    `splits` modelin CV'sinde kullanılan fold listesidir; aynı fold'lar kullanılmazsa
    doğrulama satırlarının hedef bilgisi eğitim özelliklerine sızar ve CV skoru şişer.
    """
    y_all = train_df[target]
    keys = train_df[col].astype(str)
    train_encoded = np.zeros(len(train_df))

    for tr_idx, val_idx in splits:
        tr_y = y_all.iloc[tr_idx]
        prior = tr_y.mean()
        stats = tr_y.groupby(keys.iloc[tr_idx]).agg(["mean", "count"])
        smooth = (stats["mean"] * stats["count"] + prior * alpha) / (stats["count"] + alpha)
        train_encoded[val_idx] = keys.iloc[val_idx].map(smooth).fillna(prior).values

    prior = y_all.mean()
    stats_full = y_all.groupby(keys).agg(["mean", "count"])
    smooth_full = (stats_full["mean"] * stats_full["count"] + prior * alpha) / (stats_full["count"] + alpha)
    test_encoded = test_df[col].astype(str).map(smooth_full).fillna(prior).values

    return train_encoded, test_encoded

In [ ]:
train = add_all_features(train)
test = add_all_features(test)

print("\nFeature engineering tamamlandı.")
print("train:", train.shape)
print("test :", test.shape)

In [ ]:
train = train.dropna(subset=[TARGET]).reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
freq_cols = [
    "patient_id",
    "clinic_id",
    "clinic_specialty",
    "booking_channel",
    "hour_group",
    "clinic_dow_combo",
    "channel_hour_combo",
    "specialty_dow_combo",
    "specialty_hour_combo",
]

freq_cols = [c for c in freq_cols if c in train.columns and c in test.columns]
train, test = add_frequency_encoding(train, test, freq_cols)

print("\nFrequency encoding columns:")
print(freq_cols)

### CV fold'ları
Fold'lar target encoding'den **önce** oluşturuluyor ki TE ve modeller aynı bölünmeyi kullansın.

In [ ]:
y = train[TARGET].astype(int).reset_index(drop=True)
use_group_cv = HAS_SGKF and ("patient_id" in train.columns)

if use_group_cv:
    print("StratifiedGroupKFold kullanılacak (group = patient_id).")
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(train, y, groups=train["patient_id"]))
else:
    print("StratifiedKFold kullanılacak.")
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(train, y))

for i, (tr_idx, val_idx) in enumerate(splits, 1):
    print(f"Fold {i}: train={len(tr_idx)}, val={len(val_idx)}, val pozitif oranı={y.iloc[val_idx].mean():.4f}")

In [ ]:
te_cols = [
    "patient_id",
    "clinic_id",
    "clinic_specialty",
    "booking_channel",
    "hour_group",
    "clinic_dow_combo",
    "channel_hour_combo",
    "specialty_dow_combo",
    "specialty_hour_combo",
]

# Grup CV'de doğrulama hastaları eğitimde hiç yok; patient_id TE'si train'de hep
# ortalamaya eşit olur ama test'te dolu gelir -> train/test uyumsuzluğu. Bu yüzden çıkarıyoruz.
if use_group_cv:
    te_cols = [c for c in te_cols if c != "patient_id"]

te_cols = [c for c in te_cols if c in train.columns and c in test.columns]

for col in te_cols:
    train[f"{col}_te"], test[f"{col}_te"] = kfold_target_encode(
        train_df=train,
        test_df=test,
        col=col,
        target=TARGET,
        splits=splits,
        alpha=20,
    )

print("OOF target encoding tamamlandı.")
print("TE columns:", te_cols)

In [ ]:
drop_cols = [
    "appointment_id",
    "patient_id",          # ham kimlik: grup CV'de genellemez, sadece ezberler
    TARGET,
    "appointment_datetime",
    "booking_datetime",
]
drop_cols = [c for c in drop_cols if c in train.columns]

num_cols = train.select_dtypes(include="number").columns.tolist()
num_cols = [c for c in num_cols if c not in drop_cols]

for col in num_cols:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

cat_cols = train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_cols = [c for c in cat_cols if c not in drop_cols]

# train ve test kategorilerini hizala
for col in cat_cols:
    tr_vals = train[col].astype("object").fillna("missing").astype(str)
    te_vals = test[col].astype("object").fillna("missing").astype(str)
    all_cats = pd.Index(tr_vals.unique()).union(pd.Index(te_vals.unique()))
    train[col] = pd.Categorical(tr_vals, categories=all_cats)
    test[col] = pd.Categorical(te_vals, categories=all_cats)

print("Missing value handling tamamlandı.")
print("Categorical columns:", cat_cols)

In [ ]:
features = [c for c in train.columns if c not in drop_cols]
missing_in_test = [c for c in features if c not in test.columns]
assert not missing_in_test, f"Test'te olmayan özellikler: {missing_in_test}"

X = train[features].reset_index(drop=True)
X_test = test[features].reset_index(drop=True)

print("Feature count:", len(features))

## Model eğitimi — LightGBM + XGBoost (+ CatBoost, kuruluysa) + Ensemble

In [ ]:
# XGBoost için kategori -> tamsayı kodu (kategoriler hizalı olduğu için train/test kodları tutarlı)
def make_xgb_compatible(df):
    df = df.copy()
    for col in df.columns:
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].cat.codes
        elif df[col].dtype == object:
            df[col] = pd.Categorical(df[col]).codes
    return df

# CatBoost kategorik sütunları string olarak ister
def make_cat_compatible(df):
    df = df.copy()
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
    return df

lgb_params = dict(
    n_estimators=3000,          # early stopping en iyi ağaç sayısını seçer
    learning_rate=0.02,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.80,
    reg_alpha=0.5,
    reg_lambda=1.5,
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

xgb_params = dict(
    n_estimators=3000,
    learning_rate=0.025,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.80,
    reg_alpha=0.3,
    reg_lambda=1.5,
    gamma=0.1,
    objective="binary:logistic",
    eval_metric="aucpr",
    early_stopping_rounds=150,  # xgboost >= 2.0: kurucu parametresi
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

cat_params = dict(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    loss_function="Logloss",
    eval_metric="PRAUC",
    early_stopping_rounds=150,
    random_seed=RANDOM_STATE,
    verbose=False,
)

# Ensemble ağırlıkları; yalnızca eğitilen modeller kullanılır ve ağırlıklar normalize edilir
BASE_WEIGHTS = {"lgb": 0.40, "xgb": 0.30, "cat": 0.30}
models = ["lgb", "xgb"] + (["cat"] if HAS_CATBOOST else [])
weights = {m: BASE_WEIGHTS[m] / sum(BASE_WEIGHTS[m] for m in models) for m in models}
print("Kullanılan modeller ve ensemble ağırlıkları:", weights)

def blend(preds):
    return sum(weights[m] * preds[m] for m in models)

oof = {m: np.zeros(len(X)) for m in models}
test_preds = {m: np.zeros(len(X_test)) for m in models}
scores = {m: [] for m in models + ["ens"]}
feature_importance_list = []

X_xgb_all = make_xgb_compatible(X)
X_test_xgb = make_xgb_compatible(X_test)
if HAS_CATBOOST:
    X_cat_all = make_cat_compatible(X)
    X_test_cat = make_cat_compatible(X_test)
    cat_feature_indices = [X.columns.get_loc(c) for c in cat_cols if c in X.columns]

for fold, (tr_idx, val_idx) in enumerate(splits, 1):
    print(f"\n{'='*20} Fold {fold} {'='*20}")
    y_train, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    fold_val = {}

    # 1) LIGHTGBM
    lgb_model = LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X.iloc[tr_idx], y_train,
        eval_set=[(X.iloc[val_idx], y_val)],
        eval_metric="average_precision",
        categorical_feature=[c for c in cat_cols if c in X.columns],
        callbacks=[early_stopping(stopping_rounds=150, verbose=False), log_evaluation(period=0)],
    )
    fold_val["lgb"] = lgb_model.predict_proba(X.iloc[val_idx])[:, 1]
    test_preds["lgb"] += lgb_model.predict_proba(X_test)[:, 1] / len(splits)
    feature_importance_list.append(pd.DataFrame({
        "feature": features,
        "importance": lgb_model.booster_.feature_importance(importance_type="gain"),
        "fold": fold,
    }))

    # 2) XGBOOST
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_xgb_all.iloc[tr_idx], y_train,
        eval_set=[(X_xgb_all.iloc[val_idx], y_val)],
        verbose=False,
    )
    fold_val["xgb"] = xgb_model.predict_proba(X_xgb_all.iloc[val_idx])[:, 1]
    test_preds["xgb"] += xgb_model.predict_proba(X_test_xgb)[:, 1] / len(splits)

    # 3) CATBOOST (kuruluysa)
    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(**cat_params)
        cat_model.fit(
            X_cat_all.iloc[tr_idx], y_train,
            eval_set=(X_cat_all.iloc[val_idx], y_val),
            cat_features=cat_feature_indices,
            use_best_model=True,
        )
        fold_val["cat"] = cat_model.predict_proba(X_cat_all.iloc[val_idx])[:, 1]
        test_preds["cat"] += cat_model.predict_proba(X_test_cat)[:, 1] / len(splits)

    # 4) Skorlar + ENSEMBLE
    for m in models:
        oof[m][val_idx] = fold_val[m]
        scores[m].append(average_precision_score(y_val, fold_val[m]))
        print(f"{m.upper():4s} Fold {fold} PR-AUC : {scores[m][-1]:.6f}")

    scores["ens"].append(average_precision_score(y_val, blend(fold_val)))
    print(f"ENS  Fold {fold} PR-AUC : {scores['ens'][-1]:.6f}")

In [ ]:
names = {"lgb": "LightGBM", "xgb": "XGBoost", "cat": "CatBoost", "ens": "Ensemble"}
ensemble_oof = blend(oof)

print("=" * 50)
print("MODEL BAZLI SONUÇLAR  (baz çizgi / pozitif oranı: %.4f)" % y.mean())
print("=" * 50)

for m in models + ["ens"]:
    oof_pred = ensemble_oof if m == "ens" else oof[m]
    print(f"\n{names[m]} CV scores   : {np.round(scores[m], 6).tolist()}")
    print(f"{names[m]} Mean CV PR-AUC: {np.mean(scores[m]):.6f} ± {np.std(scores[m]):.6f}")
    print(f"{names[m]} OOF PR-AUC    : {average_precision_score(y, oof_pred):.6f}")

In [ ]:
feature_importance = (
    pd.concat(feature_importance_list, axis=0)
    .groupby("feature")["importance"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("Top 40 LightGBM feature importance (gain):")
print(feature_importance.head(40).to_string())

In [ ]:
# =========================================================
# SUBMISSIONS — appointment_id üzerinden eşleştirme
# =========================================================
id_col = "appointment_id"
pred_col = [c for c in sample_sub.columns if c != id_col][0]   # sample_sub'daki hedef sütun adı

def make_submission(preds, path):
    pred_df = pd.DataFrame({id_col: test[id_col].values, pred_col: preds})
    sub = sample_sub[[id_col]].merge(pred_df, on=id_col, how="left")
    assert sub[pred_col].notna().all(), "Bazı appointment_id'ler için tahmin yok"
    sub.to_csv(path, index=False)
    print(f"- {path}  ({sub.shape[0]} satır)")

print("Oluşturulan dosyalar:")
file_names = {"lgb": "submission_lgbm_enhanced.csv",
              "xgb": "submission_xgb_enhanced.csv",
              "cat": "submission_catboost_enhanced.csv"}
for m in models:
    make_submission(test_preds[m], file_names[m])
make_submission(blend(test_preds), "submission_ensemble_enhanced.csv")